In [40]:
import pandas as pd
import numpy as np
import os
import re
import gzip
from tqdm import tqdm

In [6]:
metadata = pd.read_csv('../code/resources/GTEx/GTEx_Analysis_v8_Annotations_SampleAttributesDS.txt', sep='\t')
metadata['tissue_id'] = metadata.SMTSD.apply(lambda x: re.sub('\)', '', re.sub('\(', '_', re.sub(
    ' ', '', x))))

<>:2: SyntaxWarning: invalid escape sequence '\)'
<>:2: SyntaxWarning: invalid escape sequence '\('
<>:2: SyntaxWarning: invalid escape sequence '\)'
<>:2: SyntaxWarning: invalid escape sequence '\('
/tmp/ipykernel_820465/3867180868.py:2: SyntaxWarning: invalid escape sequence '\)'
  metadata['tissue_id'] = metadata.SMTSD.apply(lambda x: re.sub('\)', '', re.sub('\(', '_', re.sub(
/tmp/ipykernel_820465/3867180868.py:2: SyntaxWarning: invalid escape sequence '\('
  metadata['tissue_id'] = metadata.SMTSD.apply(lambda x: re.sub('\)', '', re.sub('\(', '_', re.sub(


In [12]:
tissues_to_sample = sorted(set([x.split('.')[0] for x in os.listdir(
    '/project/yangili1/cfbuenabadn/SpliFi/code/resources/GTEx/juncs/all49tissues/')]))

In [20]:
tissue_dict = {}
for tissue in tissues_to_sample:
    samples = list(metadata.loc[metadata.tissue_id == tissue].SAMPID)
    tissue_dict.update({tissue:{'samples':samples}})

In [45]:
with gzip.open('/project2/mstephens/cfbuenabadn/gtex-stm/code/gtex_tables/GTEx_Analysis_2017-06-05_v8_RNASeQCv1.1.9_gene_reads.gct.gz', 'rb') as fh:
    fh.readline()
    fh.readline()
    header = fh.readline().decode().rstrip().split('\t')
    linea2 = fh.readline().decode().rstrip().split('\t')

In [46]:
tissue_dict = {}
for tissue in tissues_to_sample:
    samples = list(metadata.loc[metadata.tissue_id == tissue].SAMPID)
    idx = [x in samples for x in header][2:]
    tissue_dict.update({tissue:{'samples':samples, 'idx':idx}})
    

In [47]:
tissue_indices = [tissue_dict[t]['idx'] for t in tissues_to_sample]

with gzip.open('GTEX_median_counts.tab.gz', 'wt') as out_fh:
    with gzip.open(
        '/project2/mstephens/cfbuenabadn/gtex-stm/code/gtex_tables/'
        'GTEx_Analysis_2017-06-05_v8_RNASeQCv1.1.9_gene_reads.gct.gz',
        'rt'
    ) as fh:

        # skip metadata
        fh.readline()
        fh.readline()

        header = fh.readline().rstrip().split('\t')
        out_fh.write('\t'.join(['gene_id', 'gene_name'] + tissues_to_sample) + '\n')

        for line in tqdm(fh):
            fields = line.rstrip().split('\t')
            init_ = fields[:2]

            vals = np.array(fields[2:], dtype=np.int64)
            counts_ = [str(np.median(vals[idx])) for idx in tissue_indices]

            out_fh.write('\t'.join(init_ + counts_) + '\n')


56200it [58:09, 16.10it/s]


In [44]:
fields

['ENSG00000223972.5',
 'DDX11L1',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '1',
 '2',
 '2',
 '0',
 '1',
 '0',
 '0',
 '3',
 '0',
 '3',
 '0',
 '0',
 '0',
 '0',
 '1',
 '0',
 '7',
 '0',
 '0',
 '1',
 '0',
 '0',
 '1',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '1',
 '24',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '2',
 '0',
 '3',
 '0',
 '0',
 '0',
 '0',
 '0',
 '1',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '1',
 '1',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '1',
 '1',
 '7',
 '0',
 '0',
 '0',
 '0',
 '0',
 '1',
 '0',
 '0',
 '0',
 '1',
 '1',
 '0',
 '0',
 '0',
 '1',
 '0',
 '1',
 '0',
 '0',
 '0',
 '0',
 '0',
 '2',
 '1',
 '0',
 '1',
 '0',
 '1',
 '0',
 '0',
 '1',
 '0',
 '0',
 '0',
 '1',
 '0',
 '0',
 '2',
 '0',
 '0',
 '0',
 '0',
 '0',
 '1',
 '0',
 '5',
 '0',
 '0',
 '1',
 '0',
 '0',
 '1',
 '0',
 '0',
 '0',
 '1',
 '2',
 '0',
 '1',
 '0',
 '0',
 '0',
 '1',
 '0',
 '0',
 '0',
 '0',
 '1',
 '19',
 '0',
 '0',
 '0',
 '1',
 '0',
 '1',
 '1',
 '0',
 '0',
 '0',
 '1',
 '1',
 '0'

In [39]:
np.median(np.array(linea2)[tissue_dict['Ovary']['idx']].astype(int))

np.float64(0.0)

In [34]:
linea2[:2]

['ENSG00000223972.5', 'DDX11L1']